In [105]:
import pandas as pd
import re
import json
from tqdm import tqdm
from jsonschema import validate as json_validate, ValidationError

In [106]:
df = pd.read_csv("processed_v2.csv")

sentences = df.groupby("sent_id")["token"].apply(list)
sentences = sentences.apply(lambda x: " ".join(x)).tolist()

print("TOTAL SENTENCES:", len(sentences))
print("SAMPLE:", sentences[:2])

TOTAL SENTENCES: 7142
SAMPLE: ['Шановні колеги , Рада національної безпеки має Секретаря Ради національної безпеки , найближчим часом будуть сформовані фактично всі підрозділи РНБО , призначені заступники Секретаря Ради національної безпеки та оборони , і я сподіваюся , що Рада національної безпеки та оборони буде відповідати тим задачам і завданням , тим викликам , які сьогодні стоять перед нашою країною .', 'Рішень , які захистять Україну , які зможуть локалізувати осередки сепаратизму , рішення , які зможуть забезпечити справедливість та покарання злочинців .']


In [107]:
def extract(text):

    return {
        "persons": re.findall(r"[А-ЯІЇЄ][а-яіїє]+ [А-ЯІЇЄ][а-яіїє]+", text),
        "orgs": re.findall(r"(Рада|ВРУ|КМУ|Міністерство|РНБО|флот)", text),
        "locations": re.findall(r"(Україна|Крим|Росія)", text),
        "legal_acts": re.findall(r"(закон|постанова|проєкт)", text),
        "dates": re.findall(r"\b(19|20)\d{2}\b|\b\d{1,2}\s\w+", text),
    }

In [108]:
schema = {
    "type": "object",
    "properties": {
        "persons": {"type": "array", "items": {"type": "string"}},
        "orgs": {"type": "array", "items": {"type": "string"}},
        "locations": {"type": "array", "items": {"type": "string"}},
        "legal_acts": {"type": "array", "items": {"type": "string"}},
        "dates": {"type": "array", "items": {"type": "string"}}
    },
    "required": ["persons","orgs","locations","legal_acts","dates"]
}

In [109]:
def validate_structure(obj):
    try:
        json_validate(instance=obj, schema=schema)
        return True
    except ValidationError:
        return False

In [110]:
def validate_semantic(obj):
    return any(len(obj[k]) > 0 for k in obj)

In [111]:
def repair(obj):
    fixed = {}

    for k in schema["properties"].keys():
        v = obj.get(k, [])
        if not isinstance(v, list):
            v = []
        fixed[k] = v

    return fixed

In [112]:
results = []

for text in tqdm(sentences):  # для Colab

    raw = extract(text)

    struct_raw = validate_structure(raw)
    sem_raw = validate_semantic(raw)

    final = raw

    if not struct_raw:
        final = repair(raw)

    struct_final = validate_structure(final)
    sem_final = validate_semantic(final)

    results.append({
        "text": text,
        "raw": raw,
        "final": final,
        "struct_raw": struct_raw,
        "sem_raw": sem_raw,
        "struct_final": struct_final,
        "sem_final": sem_final
    })

100%|██████████| 7142/7142 [01:01<00:00, 115.75it/s]


In [113]:
struct_raw_rate = sum(r["struct_raw"] for r in results) / len(results)
struct_final_rate = sum(r["struct_final"] for r in results) / len(results)

In [114]:
sem_raw_rate = sum(r["sem_raw"] for r in results) / len(results)
sem_final_rate = sum(r["sem_final"] for r in results) / len(results)

In [115]:
print("STRUCT RAW:", struct_raw_rate)
print("STRUCT FINAL:", struct_final_rate)
print("SEM RAW:", sem_raw_rate)
print("SEM FINAL:", sem_final_rate)

STRUCT RAW: 1.0
STRUCT FINAL: 1.0
SEM RAW: 0.3116774012881546
SEM FINAL: 0.3116774012881546


In [116]:
bad_cases = []

for r in results:
    if not r["sem_final"]:
        bad_cases.append(r)

for i, case in enumerate(bad_cases[:15]):
    print("="*80)
    print(case["text"])
    print("RAW:", case["raw"])
    print("FINAL:", case["final"])

Рішень , які захистять Україну , які зможуть локалізувати осередки сепаратизму , рішення , які зможуть забезпечити справедливість та покарання злочинців .
RAW: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
FINAL: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
А крім морських сил у нас немає ніяких сухопутних ?
RAW: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
FINAL: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
Тобто всіх , які нам підпорядковані .
RAW: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
FINAL: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
Але це - номінальна чисельність .
RAW: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
FINAL: {'persons': [], 'orgs': [], 'locations': [], 'legal_acts': [], 'dates': []}
Боєздатна складова ?
RAW: {'persons': [], 'orgs': [], 'locations'